# Maia Lite Chat 0.3.0 — Corpus Acquisition v8.6
## Scale-up from the validated 100M proxy-token run

**Training compute budget:** 21,289,390,080 *actual tokenizer tokens processed*. With three average passes, the initial unique-token target is ~7.10B actual tokens. **8.2B character-based proxy tokens is an acquisition planning target, NOT a guaranteed 7.10B tokenizer tokens.** Calibrate against the final tokenizer and adjust the target. This notebook reuses the six validated sources and the committed chunks from v8.5.

**Rights:** acquired documents remain quarantined, `eligible_for_training=False`. Review rights, dataset versions, provenance and restrictions before training. The notebook does not authorize use of the content.

**Operational safeguards:** incremental per-session budgets; exact-hash deduplication; crash-safe chunks; optional local SSD staging with Drive sync; immutable committed chunks; resumption reports. Large-scale near-duplicate filtering and source diversity remain separate tasks.

**Important:** `fineweb-edu/sample-10BT` is a bounded sample, and may be exhausted before a multi-billion-token quota is met. Do not fabricate or repeatedly count the same data; add additional reviewed sources/configs as needed. Reopening streams may re-scan earlier rows; no claim of O(1) stream resume.

In [ ]:
%pip -q install "datasets>=3.0,<5" "huggingface_hub>=0.26" "pandas>=2.0" "tqdm>=4.60"

In [ ]:
import os, json, gzip, hashlib, re, random, time, shutil, sqlite3, math
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict, Counter
import pandas as pd
from datasets import load_dataset, get_dataset_config_names, get_dataset_split_names
from huggingface_hub import HfApi

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab; configure ROOT manually if needed.')

ROOT = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
RUN_ID = 'acquisition_v8_5_001'  # REUSE validated 100M; do not change without intention
RUN_DIR = ROOT / 'corpus_v8' / 'runs' / RUN_ID
DRIVE_CANDIDATES = RUN_DIR / 'quarantine_pending_rights_review'
REPORTS = RUN_DIR / 'reports'
DRIVE_CANDIDATES.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

# Set to True to write new chunks on the ephemeral Colab SSD and upload to Drive
# after each atomic chunk. Never count an unsynced local chunk as committed.
USE_LOCAL_SSD_STAGING = True
LOCAL_STAGE = Path('/content/maia_corpus_staging')
LOCAL_STAGE.mkdir(parents=True, exist_ok=True)
CANDIDATES = DRIVE_CANDIDATES  # committed truth; do not store permanent data only on SSD

DRY_RUN = True  # inspect first; set False to acquire
TARGET_PROXY_TOKENS = 8_200_000_000  # provisional; recalibrate with actual tokenizer
SEED = 20261018
SHUFFLE_BUFFER = 1000
MAX_SECONDS_PER_JOB = 3600  # each Colab session is deliberately bounded
MAX_ROWS_SCANNED_PER_JOB = 2_000_000
MAX_TEXT_CHARS = 100_000
MIN_TEXT_CHARS = 250
CHUNK_DOCS = 500
MAX_DRIVE_GB = 250.0  # guardrail, not prediction; check Drive quota first
MIN_FREE_LOCAL_GB = 5.0
RUN_STREAMING = True
# Limits NEW work per execution. Increase gradually after checking storage/time.
MAX_NEW_PROXY_TOKENS_PER_SESSION = 100_000_000
MAX_NEW_PROXY_TOKENS_PER_JOB = 25_000_000

assert TARGET_PROXY_TOKENS > 0
assert MAX_NEW_PROXY_TOKENS_PER_SESSION > 0
print('Run:', RUN_DIR, 'Dry run:', DRY_RUN)
print('Existing corpus will be reused; no overwrite of committed chunks.')


## 1. Six explicit acquisition jobs

The 100M pilot uses **web 80% / Wikipedia 20%** to fill the available six language/source cells. This is deliberately different from a proposed final corpus mixture; it does not pretend that dialogue/logic/creative sources are ready.

**Important:** the target is a *ceiling* for candidate data, not a promise that the remote services can deliver all of it within runtime and storage limits.


In [ ]:
LANG_SHARES = {'pt': 0.40, 'en': 0.35, 'es': 0.25}
FAMILY_SHARES = {'web': 0.80, 'wikipedia': 0.20}
JOBS = [
    {'id':'fineweb2_pt','dataset':'HuggingFaceFW/fineweb-2','config':'por_Latn','lang':'pt','family':'web'},
    {'id':'fineweb2_es','dataset':'HuggingFaceFW/fineweb-2','config':'spa_Latn','lang':'es','family':'web'},
    {'id':'fineweb_edu_en','dataset':'HuggingFaceFW/fineweb-edu','config':'sample-10BT','lang':'en','family':'web'},
    {'id':'wikipedia_pt','dataset':'wikimedia/wikipedia','config':'20231101.pt','lang':'pt','family':'wikipedia'},
    {'id':'wikipedia_en','dataset':'wikimedia/wikipedia','config':'20231101.en','lang':'en','family':'wikipedia'},
    {'id':'wikipedia_es','dataset':'wikimedia/wikipedia','config':'20231101.es','lang':'es','family':'wikipedia'},
]
for job in JOBS:
    job['target_proxy_tokens'] = round(TARGET_PROXY_TOKENS * LANG_SHARES[job['lang']] * FAMILY_SHARES[job['family']])
print(pd.DataFrame(JOBS)[['id','lang','family','target_proxy_tokens']].to_string(index=False))
assert sum(j['target_proxy_tokens'] for j in JOBS) == TARGET_PROXY_TOKENS


## 2. Verify Hub metadata, configurations and split availability

Do **not** pass `trust_remote_code` to `get_dataset_config_names` or `get_dataset_split_names`; it causes compatibility errors in the Colab `datasets` environment seen in v8.4.


In [ ]:
api = HfApi()
source_metadata = {}
job_checks = []
for dataset_id in sorted(set(j['dataset'] for j in JOBS)):
    entry = {'dataset': dataset_id, 'sha': None, 'license_claim': None, 'configs': [], 'error': None}
    try:
        info = api.dataset_info(dataset_id)
        entry['sha'] = info.sha
        entry['license_claim'] = (info.cardData or {}).get('license')
        entry['gated'] = info.gated
    except Exception as e:
        entry['error'] = f'Hub metadata: {type(e).__name__}: {str(e)[:250]}'
    try:
        entry['configs'] = get_dataset_config_names(dataset_id)
    except Exception as e:
        entry['error'] = (entry['error'] or '') + f' Config discovery: {type(e).__name__}: {str(e)[:250]}'
    source_metadata[dataset_id] = entry
for job in JOBS:
    entry = source_metadata[job['dataset']]
    valid = job['config'] in entry['configs']
    check = {'job_id': job['id'], 'config_valid': valid, 'split': None, 'error': None}
    if valid:
        try:
            splits = get_dataset_split_names(job['dataset'], config_name=job['config'])
            check['split'] = 'train' if 'train' in splits else (splits[0] if splits else None)
            if check['split'] is None: check['error'] = 'No split available'
        except Exception as e:
            check['error'] = f'{type(e).__name__}: {str(e)[:250]}'
    else:
        check['error'] = f'Config not available: {job["config"]}'
    job_checks.append(check)
    print(job['id'], 'config:', valid, 'split:', check['split'], 'error:', check['error'])
check_by_id = {c['job_id']: c for c in job_checks}
(REPORTS / 'source_metadata.json').write_text(json.dumps(source_metadata, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
(REPORTS / 'job_checks.json').write_text(json.dumps(job_checks, indent=2, ensure_ascii=False), encoding='utf-8')


## 3. Crash-safe committed chunks and resumable deduplication

Each compressed chunk is written to a temporary file, then atomically renamed to a final filename. On restart, we rebuild the exact-deduplication index and counters from **committed final chunks**, so interrupted writes do not create phantom progress. Chunk sizes are deliberately small for Drive.

A dataset's streaming iterator may restart from the beginning when Colab reconnects; exact deduplication prevents duplicate storage, but rescanning can consume time. A checkpointed dataset iterator is **not** claimed.


In [ ]:
def utcnow():
    return datetime.now(timezone.utc).isoformat()

def normalize_text(text):
    return re.sub(r'\s+', ' ', text.casefold()).strip()

def sha256_text(text):
    return hashlib.sha256(normalize_text(text).encode('utf-8')).hexdigest()

def token_proxy(text):
    return max(1, round(len(text)/4))

def extract_text(row):
    for key in ('text', 'content', 'body', 'document', 'raw_content'):
        value = row.get(key)
        if isinstance(value, str) and value.strip():
            return value, key
    return None, None

def committed_chunks(job_id):
    return sorted(DRIVE_CANDIDATES.glob(f'{job_id}_chunk_*.jsonl.gz'))

def reconstruct_state():
    hashes = set()
    stats = {j['id']: {'stored_docs':0,'proxy_tokens':0,'bytes_uncompressed':0,'chunks':0} for j in JOBS}
    corrupt = []
    for job in JOBS:
        for file in committed_chunks(job['id']):
            try:
                with gzip.open(file, 'rt', encoding='utf-8') as handle:
                    for line in handle:
                        row = json.loads(line)
                        key = row['normalized_sha256']
                        if key in hashes:
                            raise ValueError('Duplicate hash across committed chunks: ' + key)
                        hashes.add(key)
                        stats[job['id']]['stored_docs'] += 1
                        stats[job['id']]['proxy_tokens'] += int(row['proxy_tokens'])
                        stats[job['id']]['bytes_uncompressed'] += int(row['text_bytes'])
                stats[job['id']]['chunks'] += 1
            except Exception as e:
                corrupt.append({'file':str(file),'error':f'{type(e).__name__}: {str(e)[:200]}'})
    if corrupt:
        raise RuntimeError('Corrupt/duplicate committed chunk(s); do not continue: ' + json.dumps(corrupt[:5]))
    return hashes, stats

known_hashes, committed_stats = reconstruct_state()
print('Recovered docs:',len(known_hashes))
print(pd.DataFrame([{'job_id':k,**v} for k,v in committed_stats.items()]).to_string(index=False))


## Acquisition in bounded sessions

All six jobs continue from the committed Drive chunks. New chunks can be compressed on the Colab SSD, then uploaded and verified before becoming committed. After a runtime restart, rerun all cells: the committed Drive chunks are the authoritative state. This protects against lost local SSD data, although a network interruption can require rereading streamed source records. For extremely large corpora, exact-hash reconstruction consumes RAM/time; migrate to a persistent SQLite/partitioned hash index once measured.

In [ ]:
def quarantine_size_bytes():
    return sum(f.stat().st_size for f in DRIVE_CANDIDATES.glob('*.jsonl.gz'))

def disk_ok():
    try:
        free = shutil.disk_usage('/content').free
        return free >= MIN_FREE_LOCAL_GB * (1024**3)
    except Exception:
        return True

def commit_chunk(job_id, batch, chunk_index):
    if not batch:
        return None
    final = DRIVE_CANDIDATES / f'{job_id}_chunk_{chunk_index:06d}.jsonl.gz'
    if final.exists():
        raise FileExistsError(f'Committed chunk already exists: {final}')
    work_dir = LOCAL_STAGE if USE_LOCAL_SSD_STAGING else DRIVE_CANDIDATES
    tmp = work_dir / f'.{job_id}_chunk_{chunk_index:06d}.{os.getpid()}.tmp'
    transfer = DRIVE_CANDIDATES / f'.{job_id}_chunk_{chunk_index:06d}.{os.getpid()}.uploading'
    try:
        with gzip.open(tmp, 'wt', encoding='utf-8', compresslevel=5) as handle:
            for record in batch:
                handle.write(json.dumps(record, ensure_ascii=False) + '\n')
        # Hash of compressed bytes, verified after copying to Drive.
        def file_sha256(path):
            h = hashlib.sha256()
            with open(path, 'rb') as f:
                for block in iter(lambda: f.read(4 * 1024 * 1024), b''):
                    h.update(block)
            return h.hexdigest()
        digest = file_sha256(tmp)
        shutil.copyfile(tmp, transfer)
        if file_sha256(transfer) != digest:
            raise IOError('Drive copy hash mismatch: ' + str(transfer))
        # Verify gzip stream and JSON before commit.
        with gzip.open(transfer, 'rt', encoding='utf-8') as handle:
            verified = sum(1 for line in handle if json.loads(line))
        if verified != len(batch):
            raise IOError(f'Unexpected row count: {verified} != {len(batch)}')
        os.replace(transfer, final)
        return final
    finally:
        if tmp.exists(): tmp.unlink()
        if transfer.exists(): transfer.unlink()

def acquire_job(job, check, global_hashes, stats, remaining_session_budget):
    jid = job['id']
    report = {'job_id':jid,'status':'not_run','rows_scanned':0,'accepted_docs_this_run':0,
              'new_proxy_tokens':0,'duplicate_rows':0,'short_rows':0,'error':None,
              'target_proxy_tokens':job['target_proxy_tokens'], 'started_utc':utcnow()}
    if DRY_RUN or not RUN_STREAMING:
        report['status'] = 'dry_run'; return report
    if check['error'] or not check['config_valid'] or not check['split']:
        report['status']='config_error';report['error']=check['error'];return report
    if remaining_session_budget <= 0:
        report['status']='session_budget_reached';return report
    if stats[jid]['proxy_tokens'] >= job['target_proxy_tokens']:
        report['status']='quota_met_previous_run';return report
    batch = []
    pending_hashes = set()
    next_chunk = max([int(f.stem.split('_')[-1].split('.')[0]) for f in committed_chunks(jid)], default=-1)+1
    # .stem for *.jsonl.gz ends in '.jsonl'; parsed index is split('.')[0].
    started=time.monotonic()
    per_job_start_tokens=stats[jid]['proxy_tokens']
    try:
        stream = load_dataset(job['dataset'], name=job['config'], split=check['split'], streaming=True)
        # Seeded shuffle randomizes accessible shards and within a bounded buffer.
        # It is not statistically representative of the entire source.
        stream = stream.shuffle(seed=SEED, buffer_size=SHUFFLE_BUFFER)
        for index, row in enumerate(stream):
            if index >= MAX_ROWS_SCANNED_PER_JOB:
                report['status']='row_budget_reached';break
            if time.monotonic()-started >= MAX_SECONDS_PER_JOB:
                report['status']='time_budget_reached';break
            if index % 500 == 0 and quarantine_size_bytes() >= MAX_DRIVE_GB * 1024**3:
                report['status']='storage_budget_reached';break
            if index % 500 == 0 and not disk_ok():
                report['status']='local_disk_low';break
            report['rows_scanned'] += 1
            text, field = extract_text(row)
            if not text or len(text)<MIN_TEXT_CHARS:
                report['short_rows'] += 1;continue
            text = text[:MAX_TEXT_CHARS]
            h=sha256_text(text)
            if h in global_hashes or h in pending_hashes:
                report['duplicate_rows'] += 1;continue
            proxy=token_proxy(text)
            current=stats[jid]['proxy_tokens']+sum(x['proxy_tokens'] for x in batch)
            if (current - per_job_start_tokens >= MAX_NEW_PROXY_TOKENS_PER_JOB
                or current - per_job_start_tokens >= remaining_session_budget):
                report['status']='session_budget_reached';break
            if current >= job['target_proxy_tokens']:
                report['status']='quota_met';break
            # Last document can exceed quota; recorded as such in manifest.
            doc_id = row.get('id') or row.get('article_id') or row.get('page_id')
            url = row.get('url') or row.get('source_url')
            record={
                'dataset_id':job['dataset'],'dataset_revision':source_metadata[job['dataset']]['sha'],
                'config':job['config'],'split':check['split'],'job_id':jid,
                'source_family':job['family'],'expected_language':job['lang'],
                'source_document_id':str(doc_id) if doc_id is not None else None,
                'source_url':str(url)[:1000] if url is not None else None,
                'text_field':field,'text':text,'text_bytes':len(text.encode('utf-8')),
                'normalized_sha256':h,'proxy_tokens':proxy,
                'rights_status':'pending_review','eligible_for_training':False,
                'acquired_utc':utcnow()
            }
            batch.append(record);pending_hashes.add(h)
            if len(batch)>=CHUNK_DOCS:
                final=commit_chunk(jid,batch,next_chunk)
                for r in batch:global_hashes.add(r['normalized_sha256'])
                stats[jid]['stored_docs']+=len(batch)
                stats[jid]['proxy_tokens']+=sum(r['proxy_tokens'] for r in batch)
                stats[jid]['bytes_uncompressed']+=sum(r['text_bytes'] for r in batch)
                stats[jid]['chunks']+=1
                report['accepted_docs_this_run']+=len(batch)
                report['new_proxy_tokens']+=sum(r['proxy_tokens'] for r in batch)
                next_chunk+=1;batch=[];pending_hashes.clear()
        else:
            report['status']='source_exhausted'
        if report['status']=='not_run':report['status']='quota_met'
    except Exception as e:
        report['status']='error';report['error']=f'{type(e).__name__}: {str(e)[:500]}'
    finally:
        # Commit buffered valid records even when a network exception interrupts the stream.
        if batch:
            try:
                commit_chunk(jid,batch,next_chunk)
                for r in batch:global_hashes.add(r['normalized_sha256'])
                stats[jid]['stored_docs']+=len(batch)
                stats[jid]['proxy_tokens']+=sum(r['proxy_tokens'] for r in batch)
                stats[jid]['bytes_uncompressed']+=sum(r['text_bytes'] for r in batch)
                stats[jid]['chunks']+=1
                report['accepted_docs_this_run']+=len(batch)
                report['new_proxy_tokens']+=sum(r['proxy_tokens'] for r in batch)
            except Exception as e:
                report['status']='commit_error';report['error']=f'{type(e).__name__}: {str(e)[:500]}'
        report['finished_utc']=utcnow()
    return report

job_reports=[]
remaining_session_budget=MAX_NEW_PROXY_TOKENS_PER_SESSION
for job in JOBS:
    print('Processing',job['id'],'quota',job['target_proxy_tokens'],
          'session budget remaining',remaining_session_budget)
    r=acquire_job(job,check_by_id[job['id']],known_hashes,committed_stats,remaining_session_budget)
    job_reports.append(r)
    remaining_session_budget=max(0,remaining_session_budget-r['new_proxy_tokens'])
    print(' ',r['status'],'new docs',r['accepted_docs_this_run'],
          'new proxy tokens',r['new_proxy_tokens'],'error',r['error'])
    (REPORTS/'last_job_reports.json').write_text(
        json.dumps(job_reports,indent=2,ensure_ascii=False),encoding='utf-8')


## 5. Reconstruct state and write auditable manifest

The final manifest always reports **candidate** counts, not training-eligible counts. The rights, quality, benchmark and tokenizer gates remain closed. The full 100M acquisition is considered complete only if all six quotas are reached.


In [ ]:
known_hashes, committed_stats = reconstruct_state()
actual_total=sum(x['proxy_tokens'] for x in committed_stats.values())
quota_rows=[]
for job in JOBS:
    s=committed_stats[job['id']]
    quota_rows.append({
        'job_id':job['id'],'language':job['lang'],'family':job['family'],
        'target_proxy_tokens':job['target_proxy_tokens'],
        'stored_proxy_tokens':s['proxy_tokens'],
        'remaining_proxy_tokens':max(0,job['target_proxy_tokens']-s['proxy_tokens']),
        'stored_docs':s['stored_docs'],'chunks':s['chunks']
    })
quota_df=pd.DataFrame(quota_rows)
display(quota_df)
quota_df.to_csv(REPORTS/'quota_progress.csv',index=False)
manifest={
    'version':'v8.6-scale-up','generated_utc':utcnow(),'run_id':RUN_ID,
    'dry_run':DRY_RUN,
    'max_new_proxy_tokens_per_session':MAX_NEW_PROXY_TOKENS_PER_SESSION,
    'local_ssd_staging':USE_LOCAL_SSD_STAGING,'target_proxy_tokens':TARGET_PROXY_TOKENS,
    'observed_candidate_proxy_tokens':actual_total,
    'candidate_docs':len(known_hashes),
    'all_quotas_met':all(r['remaining_proxy_tokens']==0 for r in quota_rows),
    'language_target_shares':LANG_SHARES,'acquisition_family_shares':FAMILY_SHARES,
    'quotas':quota_rows,'job_reports':job_reports,
    'quarantine_storage_bytes':quarantine_size_bytes(),
    'sample_method':'seeded streaming shard/buffer shuffle; not dataset-wide probability sample',
    'dedup_scope':'normalized SHA256 exact only across committed v8.5 documents',
    'token_estimator':'round(characters/4), not Maia tokenizer',
    'source_metadata_file':str(REPORTS/'source_metadata.json'),
    'rights_status':'pending_review','eligible_training_tokens':0,
    'production_ready':False,
    'blockers':[
        'Human source/document license and rights review not completed',
        'No final tokenizer; proxy tokens are not actual model tokens',
        'Near-duplicate and multilingual quality audits not performed at full scale',
        'No frozen benchmark contamination audit',
        'Dialogue, logic, creative and other source families still missing',
        'Shard/buffer shuffle is not statistically representative'
    ],
    'outputs':['reports/source_metadata.json','reports/job_checks.json','reports/quota_progress.csv',
               'reports/last_job_reports.json','reports/manifest_v8_6_scaleup.json',
               'quarantine_pending_rights_review/*_chunk_*.jsonl.gz']
}
manifest_path=REPORTS/'manifest_v8_6_scaleup.json'
manifest_path.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
print('Manifest:',manifest_path)
print('Candidate proxy tokens:',actual_total)
print('All quotas met:',manifest['all_quotas_met'])
print('Production ready:',manifest['production_ready'])


## How to run safely

1. Run with `DRY_RUN=True` to inspect quotas and recover the 100M already committed. Confirm available Drive space.
2. Set `DRY_RUN=False`, `MAX_NEW_PROXY_TOKENS_PER_SESSION=100_000_000` for the first scale-up session; increase only after checking elapsed time and storage.
3. At every Colab restart, mount Drive and rerun cells. Do **not** delete the Drive run directory. Local SSD staging is disposable.
4. Review `reports/manifest_v8_6_scaleup.json` and `quota_progress.csv`. `All quotas met=False` is expected until the large target is reached.
5. Check each source for exhaustion. The current six sources are **not** a guarantee of sufficient licensed, distinct data for 8.2B proxy tokens.
6. Before pretraining, run the **actual Maia tokenizer** on a stratified sample by source/language. Compute `actual_tokens / proxy_tokens`, estimate the corpus-wide actual token count and adjust target. Decide sampling/repeat schedule so processed actual tokens total 21,289,390,080.
7. During training, copy the finalized approved corpus from Drive to the Colab SSD at every new runtime (~7 planned sessions), verify hashes, and separately restore model/optimizer/scheduler checkpoints. This acquisition notebook does **not** automatically stage the entire multi-billion-token corpus to SSD.

**Note:** This implementation reuses the v8.5 exact-hash set in RAM and rereads Drive chunks on startup; at billion-token scale, measure startup cost and migrate to an indexed manifest / persistent dedup store if needed. Acquisition data remain in rights-review quarantine.